# Pandas I-III — Ejercicios Complemento Codex — Soluciones

#### Nota Codex — Cómo trabajar

Estos 15 ejercicios siguen la progresión I → II → III y terminan con una síntesis. Ejecuta Preparación al iniciar. Cada ejercicio usa los datos de preparación o crea los suyos; no depende de respuestas anteriores. Intenta resolverlo antes de consultar el cuaderno de soluciones. Los assert de las soluciones verifican resultados concretos y no sustituyen su interpretación.

## Preparación

#### Nota Codex — Datos y dependencias

Se generan todos los datos. Se requiere pandas y openpyxl para Excel; sqlite3 forma parte de Python. Los archivos se crean en una carpeta temporal y se eliminan al ejecutar Cierre. Guarda tus respuestas en una copia del cuaderno.

In [1]:
import pandas as pd
from IPython.display import display
from io import StringIO
from pathlib import Path
from tempfile import TemporaryDirectory
import sqlite3

temporal = TemporaryDirectory(prefix='ejercicios_pandas_codex_')
carpeta = Path(temporal.name)
pedidos = pd.DataFrame({
    'cliente': ['Ana', 'Luis', 'Ana', 'Marta'],
    'ciudad': ['Lima', 'Quito', 'Lima', 'Bogotá'],
    'unidades': [2, 5, 1, 4],
    'precio': [10.0, 8.0, 12.0, 9.0]
}, index=[10, 20, 40, 80])
display(pedidos)

,cliente,ciudad,unidades,precio
10,Ana,Lima,2,10.0
20,Luis,Quito,5,8.0
40,Ana,Lima,1,12.0
80,Marta,Bogotá,4,9.0


## Ejercicio 1: Estructura y dimensiones

**Pandas I · básico**

### Ejercicio a analizar

Extrae precio como Series y como DataFrame. Muestra el tipo, ndim y shape de ambos, así como len y size de pedidos. Explica por qué size y len son distintos. Convierte la Series a un array e indica qué información se pierde.

Referencia: complemento I, secciones 1 y 2. No cambies pedidos.

In [2]:
# Solución Codex
serie = pedidos['precio']
tabla = pedidos[['precio']]
print(type(serie).__name__, serie.ndim, serie.shape)
print(type(tabla).__name__, tabla.ndim, tabla.shape)
print('Filas:', len(pedidos), 'Celdas:', pedidos.size)
print(serie.to_numpy())
assert serie.shape == (4,) and tabla.shape == (4, 1)
assert len(pedidos) == 4 and pedidos.size == 16

Series 1 (4,)
DataFrame 2 (4, 1)
Filas: 4 Celdas: 16
[10.  8. 12.  9.]


#### Nota Codex — Solución explicada

La Series tiene un eje y la tabla tiene dos. pedidos contiene 4 filas y 16 celdas de datos; su índice no cuenta como columna. El array conserva los precios pero pierde etiquetas y nombre.

## Ejercicio 2: Mostrar seis filas sin perder datos

**Pandas I · básico**

### Ejercicio a analizar

Crea una Series con los enteros de 0 a 19. Usa un contexto de opciones con display.max_rows=10 y display.min_rows=6 para mostrarla truncada. Después muestra head(6). Comprueba que la Series original sigue teniendo 20 elementos.

Pista: `with pd.option_context('display.max_rows', 10, 'display.min_rows', 6):` aplica las opciones solo dentro del bloque. Referencia: Nota Codex del curso I sobre mostrar y recortar.

In [3]:
# Solución Codex
numeros = pd.Series(range(20))
with pd.option_context('display.max_rows', 10, 'display.min_rows', 6):
    display(numeros)
display(numeros.head(6))
assert len(numeros) == 20

0      0
1      1
2      2
      ..
17    17
18    18
19    19
Length: 20, dtype: int64

0    0
1    1
2    2
3    3
4    4
5    5
dtype: int64

#### Nota Codex — Solución explicada

La salida truncada muestra el comienzo y el final; head(6) devuelve los valores 0 a 5. Ninguna de las dos acciones borra registros. Al salir de with se restauran las opciones anteriores.

## Ejercicio 3: Alinear antes de sumar

**Pandas I · intermedio**

### Ejercicio a analizar

Construye dos Series: lunes con A=10 y B=20; martes con B=3 y A=4, en ese orden. Antes de sumar, escribe los totales esperados. Compara la suma de Series con la de sus arrays. Después construye una tabla con ambas Series.

Referencia: complemento I, secciones 3 y 4.

In [4]:
# Solución Codex
lunes = pd.Series([10, 20], index=['A', 'B'])
martes = pd.Series([3, 4], index=['B', 'A'])
print('Esperado: A=14, B=23')
display(lunes + martes)
print('Arrays:', lunes.to_numpy() + martes.to_numpy())
display(pd.DataFrame({'lunes': lunes, 'martes': martes}))
assert (lunes + martes).to_dict() == {'A': 14, 'B': 23}

Esperado: A=14, B=23
Arrays: [13 24]


A    14
B    23
dtype: int64

,lunes,martes
A,10,4
B,20,3


#### Nota Codex — Solución explicada

Los totales de Pandas respetan A y B. Los arrays dan [13, 24] porque suman posiciones. La tabla conserva la correspondencia de etiquetas aun cuando las Series llegan en órdenes diferentes.

## Ejercicio 4: Desconocido no es cero

**Pandas I · intermedio**

### Ejercicio a analizar

Crea cantidades [3, desconocido, 0] con dtype Int64 y códigos ["001", "010", desconocido] con dtype string. Construye una tabla con ellas y explica por qué el segundo valor de cantidades no puede interpretarse como cero.

Referencia: complemento I, sección 5. No conviertas ni rellenes los faltantes.

In [5]:
# Solución Codex
cantidades = pd.Series([3, pd.NA, 0], dtype='Int64')
codigos = pd.Series(['001', '010', pd.NA], dtype='string')
tabla_tipos = pd.DataFrame({'codigo': codigos, 'cantidad': cantidades})
display(tabla_tipos, tabla_tipos.dtypes)
assert pd.isna(tabla_tipos.loc[1, 'cantidad'])
assert tabla_tipos.loc[2, 'cantidad'] == 0

,codigo,cantidad
0,001,3
1,010,<NA>
2,<NA>,0


codigo      string[python]
cantidad             Int64
dtype: object

#### Nota Codex — Solución explicada

El segundo valor no fue informado; el tercero sí fue informado y es cero. string conserva los ceros de los códigos. Aquí se representa la ausencia; decidir su tratamiento vendrá en Pandas IV y V.

## Ejercicio 5: Límites y estructura de salida

**Pandas II · básico**

### Ejercicio a analizar

Sobre pedidos, extrae: la segunda fila; una tabla con la etiqueta 20; las etiquetas 10 a 40 inclusive; las primeras dos filas y las columnas ciudad y unidades. Escribe cuántas filas tendrá cada salida antes de ejecutar.

Referencia: complemento II, secciones 1 y 2.

In [6]:
# Solución Codex
display(pedidos.iloc[1])
display(pedidos.loc[[20]])
display(pedidos.loc[10:40])
display(pedidos.iloc[:2].loc[:, ['ciudad', 'unidades']])
assert len(pedidos.loc[10:40]) == 3
assert len(pedidos.iloc[:2]) == 2

cliente      Luis
ciudad      Quito
unidades        5
precio        8.0
Name: 20, dtype: object

,cliente,ciudad,unidades,precio
20,Luis,Quito,5,8.0


,cliente,ciudad,unidades,precio
10,Ana,Lima,2,10.0
20,Luis,Quito,5,8.0
40,Ana,Lima,1,12.0


,ciudad,unidades
10,Lima,2
20,Quito,5


#### Nota Codex — Solución explicada

Las dos primeras selecciones representan una fila, pero una es Series y otra DataFrame. loc[10:40] devuelve tres filas y iloc[:2] devuelve dos. La selección encadenada aquí solo lee; para asignar usaríamos una sola operación loc.

## Ejercicio 6: Condiciones y ausencia de coincidencias

**Pandas II · intermedio**

### Ejercicio a analizar

Selecciona ciudad y unidades de pedidos que estén en Lima o Quito y tengan de 2 a 5 unidades inclusive. Usa isin y between. Selecciona después los pedidos que NO están en esas ciudades. Finalmente verifica si hay pedidos de más de 100 unidades sin intentar acceder a la primera fila de una tabla vacía.

Referencia: complemento II, sección 3.

In [7]:
# Solución Codex
en_ciudades = pedidos['ciudad'].isin(['Lima', 'Quito'])
en_intervalo = pedidos['unidades'].between(2, 5)
resultado = pedidos.loc[en_ciudades & en_intervalo, ['ciudad', 'unidades']]
display(resultado)
display(pedidos.loc[~en_ciudades])
print('Sin pedidos mayores a 100:', pedidos.loc[pedidos['unidades'] > 100].empty)
assert resultado.index.tolist() == [10, 20]

Sin pedidos mayores a 100: True


,ciudad,unidades
10,Lima,2
20,Quito,5


,cliente,ciudad,unidades,precio
80,Marta,Bogotá,4,9.0


#### Nota Codex — Solución explicada

Se seleccionan 10 y 20; la etiqueta 40 queda fuera porque tiene una unidad. La negación devuelve 80. Una tabla vacía es una respuesta válida; acceder a iloc[0] en ella produciría un error.

## Ejercicio 7: Máscara reordenada y copia independiente

**Pandas II · intermedio**

### Ejercicio a analizar

Crea una máscara para ciudad igual a Lima y cambia su orden con iloc[::-1]. Filtra pedidos usando esa Series en loc: ¿qué etiquetas quedan? Luego crea una copia de pedidos y cambia a 9 las unidades de Lima solo en la copia, con una única asignación loc. Comprueba que pedidos no cambió.

Referencia: complemento II, secciones 3 y 4.

In [8]:
# Solución Codex
mascara = (pedidos['ciudad'] == 'Lima').iloc[::-1]
display(pedidos.loc[mascara])
copia = pedidos.copy()
copia.loc[copia['ciudad'] == 'Lima', 'unidades'] = 9
display(copia)
assert pedidos.loc[mascara].index.tolist() == [10, 40]
assert pedidos['unidades'].tolist() == [2, 5, 1, 4]

,cliente,ciudad,unidades,precio
10,Ana,Lima,2,10.0
40,Ana,Lima,1,12.0


,cliente,ciudad,unidades,precio
10,Ana,Lima,9,10.0
20,Luis,Quito,5,8.0
40,Ana,Lima,9,12.0
80,Marta,Bogotá,4,9.0


#### Nota Codex — Solución explicada

La máscara conserva sus etiquetas, por lo que loc selecciona 10 y 40 aunque su orden se invierta. La copia expresa que queremos una variante independiente. Una asignación como copia = pedidos solo crearía un alias.

## Ejercicio 8: Índice repetido y selección por niveles

**Pandas II · desafío**

### Ejercicio a analizar

Crea una tabla con cliente [A, A, B], producto [libro, lápiz, libro] y unidades [2, 3, 1]. Usa cliente como índice y selecciona A. Luego crea un MultiIndex cliente-producto a partir de la tabla original y extrae todas las compras de libro con xs. Explica por qué seleccionar una etiqueta no siempre devuelve una sola fila.

Referencia: complemento II, sección 5 y MultiIndex del curso.

In [9]:
# Solución Codex
compras = pd.DataFrame({'cliente': ['A', 'A', 'B'],
                        'producto': ['libro', 'lápiz', 'libro'],
                        'unidades': [2, 3, 1]})
por_cliente = compras.set_index('cliente')
display(por_cliente.loc['A'])
multi = compras.set_index(['cliente', 'producto'])
libros = multi.xs('libro', level='producto')
display(libros)
assert len(por_cliente.loc['A']) == 2
assert libros['unidades'].tolist() == [2, 1]

,producto,unidades
cliente,,
A,libro,2
A,lápiz,3


,unidades
cliente,
A,2
B,1


#### Nota Codex — Solución explicada

A se repite y devuelve dos registros. La pareja cliente-producto es única en este ejemplo. xs selecciona un nivel y, por defecto, lo retira del índice de salida. No se realiza ninguna agregación.

## Ejercicio 9: No perder la primera fila

**Pandas III · básico**

### Ejercicio a analizar

Lee este texto separado por punto y coma, sin cabecera: `001;Ana;2
010;Luis;5
`. Asigna los nombres codigo, cliente y unidades. Conserva codigo como texto. Comprueba que se cargaron dos registros y que el primero tiene código 001.

Referencia: complemento III, sección 1.

In [10]:
# Solución Codex
texto_sin_cabecera = '001;Ana;2\n010;Luis;5\n'
sin_cabecera = pd.read_csv(StringIO(texto_sin_cabecera), sep=';', header=None,
                          names=['codigo', 'cliente', 'unidades'],
                          dtype={'codigo': 'string'})
display(sin_cabecera)
assert len(sin_cabecera) == 2
assert sin_cabecera.loc[0, 'codigo'] == '001'

,codigo,cliente,unidades
0,001,Ana,2
1,010,Luis,5


#### Nota Codex — Solución explicada

header=None impide consumir el primer registro como cabecera. dtype protege el código. header=0 con names sería apropiado para reemplazar nombres existentes, pero aquí perdería a Ana.

## Ejercicio 10: Interpretar el contrato de origen

**Pandas III · intermedio**

### Ejercicio a analizar

Genera y lee este CSV en memoria:

```text
codigo;zona;importe;fecha
001;NA;1.250,50;02/04/2026
010;SUR;SIN_DATO;03/04/2026
```

La fuente confirma que NA es una zona válida, SIN_DATO representa importe desconocido y las fechas están en día/mes/año. Conserva los códigos, interpreta el importe y las fechas y comprueba cuatro valores concretos.

Referencia: complemento III, sección 2.

In [11]:
# Solución Codex
texto_contrato = 'codigo;zona;importe;fecha\n001;NA;1.250,50;02/04/2026\n010;SUR;SIN_DATO;03/04/2026\n'
contrato = pd.read_csv(StringIO(texto_contrato), sep=';', decimal=',', thousands='.',
                       dtype={'codigo': 'string', 'zona': 'string'},
                       keep_default_na=False, na_values={'importe': ['SIN_DATO']},
                       parse_dates=['fecha'], date_format='%d/%m/%Y')
display(contrato)
assert contrato.loc[0, 'codigo'] == '001'
assert contrato.loc[0, 'zona'] == 'NA'
assert contrato.loc[0, 'importe'] == 1250.5
assert contrato.loc[0, 'fecha'] == pd.Timestamp('2026-04-02')
assert pd.isna(contrato.loc[1, 'importe'])

,codigo,zona,importe,fecha
0,001,NA,1250.5,2026-04-02
1,010,SUR,NaN,2026-04-03


#### Nota Codex — Solución explicada

Cada parámetro corresponde a una regla documentada. keep_default_na=False conserva NA, mientras na_values reconoce SIN_DATO solo en importe. No se rellena el importe desconocido ni se hace un diagnóstico de calidad completo.

## Ejercicio 11: Columnas solicitadas y campo entre comillas

**Pandas III · intermedio**

### Ejercicio a analizar

Genera este texto:

```text
codigo;comentario;unidades
001;"Entrega; por la tarde";2
010;Normal;5
```

Lee únicamente codigo y comentario del primer registro. Devuelve las columnas en el orden comentario, codigo. Comprueba que el comentario conserva el punto y coma y que la tabla tiene forma (1, 2).

Referencia: complemento III, secciones 3 y 4.

In [12]:
# Solución Codex
texto_citado = 'codigo;comentario;unidades\n001;"Entrega; por la tarde";2\n010;Normal;5\n'
parcial = pd.read_csv(StringIO(texto_citado), sep=';', nrows=1,
                     usecols=['codigo', 'comentario'], dtype={'codigo': 'string'})
parcial = parcial.loc[:, ['comentario', 'codigo']]
display(parcial)
assert parcial.shape == (1, 2)
assert parcial.loc[0, 'comentario'] == 'Entrega; por la tarde'

,comentario,codigo
0,Entrega; por la tarde,001


#### Nota Codex — Solución explicada

Las comillas protegen el separador del comentario. usecols reduce la lectura y loc fija el orden deseado. nrows limita la carga; head aplicado después no evitaría leer el archivo completo.

## Ejercicio 12: Ida y vuelta CSV

**Pandas III · intermedio**

### Ejercicio a analizar

Crea una tabla nueva con codigo [001, 010] de tipo string y unidades [2, 5]. Expórtala a carpeta / "ida_vuelta.csv" sin índice, con punto y coma y UTF-8. Reléela preservando los códigos y comprueba que coincide con la original.

Referencia: complemento III, sección 5. No necesitas completar los ejercicios anteriores para resolver este.

In [13]:
# Solución Codex
origen = pd.DataFrame({'codigo': pd.Series(['001', '010'], dtype='string'),
                       'unidades': [2, 5]})
ruta = carpeta / 'ida_vuelta.csv'
origen.to_csv(ruta, sep=';', index=False, encoding='utf-8')
vuelta = pd.read_csv(ruta, sep=';', encoding='utf-8', dtype={'codigo': 'string'})
pd.testing.assert_frame_equal(origen, vuelta)
display(vuelta)

,codigo,unidades
0,001,2
1,010,5


#### Nota Codex — Solución explicada

La verificación compara estructura, datos y tipos. El índice automático no se guardó porque no aporta un identificador del negocio. Si el índice fuera un código significativo, habría que conservarlo o pasarlo a columna explícitamente.

## Ejercicio 13: Elegir una hoja Excel

**Pandas III · intermedio**

### Ejercicio a analizar

Crea carpeta / "practica.xlsx" con dos hojas: pedidos (una tabla codigo=[001, 010] y unidades=[2, 5]) y notas (una columna texto con "Práctica"). Consulta los nombres de las hojas, importa pedidos conservando los códigos y explica qué devolvería sheet_name=None.

Referencia: complemento III, sección 6. Requiere openpyxl.

In [14]:
# Solución Codex
datos_excel = pd.DataFrame({'codigo': ['001', '010'], 'unidades': [2, 5]})
ruta_excel = carpeta / 'practica.xlsx'
with pd.ExcelWriter(ruta_excel, engine='openpyxl') as escritor:
    datos_excel.to_excel(escritor, sheet_name='pedidos', index=False)
    pd.DataFrame({'texto': ['Práctica']}).to_excel(escritor, sheet_name='notas', index=False)
with pd.ExcelFile(ruta_excel, engine='openpyxl') as libro:
    print(libro.sheet_names)
    importado = pd.read_excel(libro, sheet_name='pedidos', dtype={'codigo': 'string'})
display(importado)
assert importado.loc[0, 'codigo'] == '001'

['pedidos', 'notas']


,codigo,unidades
0,001,2
1,010,5


#### Nota Codex — Solución explicada

Se elige pedidos por nombre, evitando depender de su posición. sheet_name=None devolvería un diccionario de DataFrames, no una tabla única. CSV y Excel requieren decisiones explícitas para preservar identificadores textuales.

## Ejercicio 14: Consultar SQLite sin unir tablas

**Pandas III · desafío**

### Ejercicio a analizar

La siguiente preparación crea una tabla local en memoria. Completa la lectura para recuperar codigo y unidades de los registros con al menos 3 unidades, ordenados por codigo, usando un parámetro SQL. Usa codigo como índice y cierra la conexión incluso si ocurre un error.

```python
conexion = sqlite3.connect(':memory:')
conexion.execute('CREATE TABLE pedidos (codigo TEXT, unidades INTEGER)')
conexion.executemany('INSERT INTO pedidos VALUES (?, ?)', [('001', 2), ('010', 5), ('020', 3)])
```

Referencia: complemento III, sección 7.

In [15]:
# Solución Codex
conexion = sqlite3.connect(':memory:')
try:
    conexion.execute('CREATE TABLE pedidos (codigo TEXT, unidades INTEGER)')
    conexion.executemany('INSERT INTO pedidos VALUES (?, ?)',
                        [('001', 2), ('010', 5), ('020', 3)])
    consulta = pd.read_sql_query(
        'SELECT codigo, unidades FROM pedidos WHERE unidades >= ? ORDER BY codigo',
        conexion, params=(3,), index_col='codigo')
    display(consulta)
    assert consulta.index.tolist() == ['010', '020']
finally:
    conexion.close()

,unidades
codigo,
010,5
020,3


#### Nota Codex — Solución explicada

La consulta devuelve 010 y 020. El umbral se entrega en params, separado del texto SQL. Esta lectura no exige joins ni integración de fuentes; esos temas quedan para Pandas VI.

## Ejercicio 15: De un archivo a una entrega verificable

**Pandas I–III · síntesis**

### Ejercicio a analizar

Una tienda entrega este archivo, con una línea de metadatos antes de la cabecera:

```text
Reporte de pedidos
codigo;ciudad;unidades;precio;fecha
001;Lima;2;10,50;02/04/2026
010;Quito;5;8,00;03/04/2026
020;Lima;1;12,00;04/04/2026
030;Bogotá;4;9,00;05/04/2026
```

1. Genera el archivo en carpeta e impórtalo conservando códigos y fechas.
2. Usa codigo como índice. Explica la diferencia entre una columna Series y DataFrame, y entre la primera posición y la etiqueta 001.
3. Selecciona pedidos de Lima o Quito con al menos 2 unidades. Conserva ciudad, unidades, precio y fecha, en ese orden.
4. Crea una copia para la entrega y cambia las unidades de 001 a 3 solo en esa copia. Comprueba que el original sigue teniendo 2.
5. Exporta la entrega conservando codigo y vuelve a importarla con tipos y fechas apropiados. Comprueba la igualdad.
6. Explica qué información habría podido perderse en una lectura o escritura automática.

Referencias: complemento I, secciones 1 y 5; II, secciones 1, 3 y 4; III, secciones 1, 2 y 5. No se requieren agregaciones, limpieza, uniones ni gráficos.

In [16]:
# Solución Codex
texto_sintesis = 'Reporte de pedidos\ncodigo;ciudad;unidades;precio;fecha\n001;Lima;2;10,50;02/04/2026\n010;Quito;5;8,00;03/04/2026\n020;Lima;1;12,00;04/04/2026\n030;Bogotá;4;9,00;05/04/2026\n'
entrada = carpeta / 'entrada_sintesis.csv'
entrada.write_text(texto_sintesis, encoding='utf-8')
base = pd.read_csv(entrada, sep=';', skiprows=1, header=0, decimal=',',
                   encoding='utf-8', dtype={'codigo': 'string', 'ciudad': 'string'},
                   parse_dates=['fecha'], date_format='%d/%m/%Y', index_col='codigo')
print('Series:', base['precio'].shape, 'DataFrame:', base[['precio']].shape)
print('Primera posición:', base.iloc[0].name, 'Etiqueta 001:', base.loc['001', 'ciudad'])
criterio = base['ciudad'].isin(['Lima', 'Quito']) & (base['unidades'] >= 2)
entrega = base.loc[criterio, ['ciudad', 'unidades', 'precio', 'fecha']].copy()
entrega.loc['001', 'unidades'] = 3
assert base.loc['001', 'unidades'] == 2
assert entrega.index.tolist() == ['001', '010']
salida = carpeta / 'entrega_sintesis.csv'
entrega.to_csv(salida, sep=';', index=True, encoding='utf-8', date_format='%Y-%m-%d')
relectura = pd.read_csv(salida, sep=';', encoding='utf-8', index_col='codigo',
                        dtype={'codigo': 'string', 'ciudad': 'string'},
                        parse_dates=['fecha'], date_format='%Y-%m-%d')
pd.testing.assert_frame_equal(entrega, relectura)
display(relectura)

Series: (4,) DataFrame: (4, 1)
Primera posición: 001 Etiqueta 001: Lima


,ciudad,unidades,precio,fecha
codigo,,,,
001,Lima,3,10.5,2026-04-02
010,Quito,5,8.0,2026-04-03


#### Nota Codex — Solución explicada

La entrega tiene dos filas, con códigos 001 y 010 y unidades 3 y 5. Las fechas son 2 y 3 de abril. El original no cambia. Una lectura automática podría perder ceros iniciales o interpretar mal la fecha; una escritura sin índice perdería codigo en esta tabla. Una exportación conserva valores solo si la relectura respeta su representación.

## Cierre

#### Nota Codex — Al terminar

Comprueba que puedes explicar tanto los valores como las etiquetas, los tipos y la cantidad de filas. Ejecuta la siguiente celda al terminar; para repetir lecturas de archivos luego, vuelve a generar los datos desde Preparación.

In [17]:
temporal.cleanup()

#### Nota Codex — Continuación del curso

El diagnóstico de nulos, duplicados, frecuencias y estadísticos se estudia en **Pandas IV**.
La conversión y limpieza de columnas, imputación, recodificación y tratamiento de texto se estudian en **Pandas V**.
`concat` y `merge` se estudian en **Pandas VI**.
Los gráficos y su preparación están en [Visualización con Pandas](../../18_VISUALIZACION/02_PANDAS/01_Notebooks/04_Visualization_II_Pandas.ipynb). Aquí no se repiten esos temas.